In [4]:
import numpy as np
import pandas as pd

from anndata import AnnData
from liana.utils import spatial_neighbors

from liana.method.sp._misty._misty_constructs import lrMistyDataByCellType
from liana.method.sp import LinearModel


# Cell-type-specific MISTy example

This notebook demonstrates the cell-type-specific MISTy setup. The object is
constructed with all three cell types, and cell type A is selected as the
receiver during fitting. B and C provide the sender-specific extra views.
The goal is to recover which sender cell type and ligand explain each
receptor in A.



In [5]:
# Synthetic single-cell dataset with one receiver type (A) and two sender types (B/C)

rng = np.random.default_rng(42)
n_per_type = 100
cell_types = np.repeat(["A", "B", "C"], n_per_type)
n_cells = len(cell_types)

# Keep all cell types in the same spatial field so each receiver can have
# neighboring cells from both sender populations.
spatial = rng.uniform(0, 100, size=(n_cells, 2))

# ligX and ligY are decoys: they are expressed by senders but do not
# generate the receptor signals below. They are included in the resource
# only as partners of other receptors so target-specific filtering matters.
genes = ["ligA", "ligB", "ligX", "ligC", "ligD", "ligY",
         "recepE", "recepF", "recepG"]
# Start without background expression so each ligand has one clear sender type.
X = np.zeros((n_cells, len(genes)), dtype=np.float32)

# Sender-specific ligand expression.
is_A = cell_types == "A"
is_B = cell_types == "B"
is_C = cell_types == "C"
X[is_B, 0:3] += rng.poisson(3, size=(is_B.sum(), 3))
X[is_C, 3:6] += rng.poisson(3, size=(is_C.sum(), 3))

# Generate receptor expression from spatially weighted sender ligands.
# This creates a signal that the MISTy paraviews should be able to recover.
connectivity_adata = AnnData(
    X=np.zeros((n_cells, 1), dtype=np.float32),
    obsm={"spatial": spatial},
)
W = spatial_neighbors(
    connectivity_adata,
    spatial_key="spatial",
    bandwidth=25,
    cutoff=0,
    set_diag=False,
    inplace=False,
)

sender_ligands = np.zeros((n_cells, 6), dtype=np.float32)
sender_ligands[is_B, 0:3] = X[is_B, 0:3]
sender_ligands[is_C, 3:6] = X[is_C, 3:6]
weighted_ligands = W @ sender_ligands

X[is_A, 6] = (
    2 * weighted_ligands[is_A, 0]
    + rng.normal(0, 0.2, size=is_A.sum())
)
X[is_A, 7] = (
    2 * weighted_ligands[is_A, 1]
    + rng.normal(0, 0.2, size=is_A.sum())
)
X[is_A, 8] = (
    2 * weighted_ligands[is_A, 3]
    + rng.normal(0, 0.2, size=is_A.sum())
)
X[:, 6:9] = np.clip(X[:, 6:9], a_min=0, a_max=None)

adata_sc = AnnData(
    X=X,
    obs=pd.DataFrame({"cell_type": pd.Categorical(cell_types)}),
    var=pd.DataFrame(index=genes),
    obsm={"spatial": spatial},
)

resource_sc = pd.DataFrame({
    # True pairs: ligA -> recepE, ligB -> recepF, ligC -> recepG.
    # Decoy pairs make other ligands available in the views, while they
    # remain unsupported for the receptor they do not generate.
    "ligand": ["ligA", "ligB", "ligC", "ligD", "ligX", "ligY"],
    "receptor": ["recepE", "recepF", "recepG", "recepF",
                  "recepE", "recepF"],
})

adata_sc


/home/leoniz/projects/liana-py/.pixi/envs/default/lib/python3.13/functools.py:934: ImplicitModificationWarning: Transforming to str index.


AnnData object with n_obs × n_vars = 300 × 9
    obs: 'cell_type'
    obsm: 'spatial'
    layers: None (.X)

## Simulation

The expression matrix contains six ligands and three receptors. B cells
express ligA, ligB and ligX; C cells express ligC, ligD and ligY. The
receptors are generated only in A cells from spatially weighted ligand
signals:

- recepE is generated from ligA
- recepF is generated from ligB
- recepG is generated from ligC

The spatial connectivity uses `set_diag=False`, so a cell does not
contribute its own expression to its receptor signal. Since the fitting
mask selects A cells, the expected result is that B explains recepE and
recepF, while C explains recepG. The corresponding sender views should
have higher importance than the other sender view for those targets.


In [6]:
misty_sc = lrMistyDataByCellType(
    adata=adata_sc,
    resource=resource_sc,
    celltype_key="cell_type",
    nz_threshold=0,
    bandwidth=25,
    cutoff=0,
)

print(misty_sc)
print("views:", misty_sc.mod.keys())
print("intra features:", misty_sc.mod["intra"].var_names.tolist())
print("extra_B features:", misty_sc.mod["extra_B"].var_names.tolist())
print("extra_C features:", misty_sc.mod["extra_C"].var_names.tolist())
print("receiver groups:", misty_sc.mod["intra"].obs["cell_type"].cat.categories.tolist())

misty_sc.obs["is_A"] = misty_sc.obs["cell_type"] == "A"
misty_sc(model=LinearModel, maskby="is_A", exclude_autocrine=True, k_cv=5)
display(misty_sc.uns["target_metrics"])

# The output should contain only resource-supported target/ligand pairs.
interactions = misty_sc.uns["interactions"]
display(interactions[interactions["view"].str.startswith("extra_")])

allowed_pairs = set(zip(resource_sc["receptor"], resource_sc["ligand"], strict=True))
extra_interactions = interactions[interactions["view"].str.startswith("extra_")]
observed_pairs = set(zip(extra_interactions["target"], extra_interactions["predictor"], strict=True))
assert observed_pairs <= allowed_pairs
print("All reported interactions are resource-supported.")


/home/leoniz/projects/liana-py/.pixi/envs/default/lib/python3.13/site-packages/mudata/_core/mudata.py:1471: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
/home/leoniz/projects/liana-py/.pixi/envs/default/lib/python3.13/site-packages/mudata/_core/mudata.py:613: UserWarning: Cannot join columns with the same name because var_names are intersecting.
/home/leoniz/projects/liana-py/.pixi/envs/default/lib/python3.13/site-packages/mudata/_core/mudata.py:1322: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
/ho

MuData object with n_obs × n_vars = 300 × 21
  obs:	'cell_type'
  uns:	'_misty_ligands_by_receptor', '_misty_by_cell_type', '_misty_celltype_key'
  4 modalities
    intra:	300 × 3
      obs:	'cell_type'
      obsm:	'spatial'
      layers:	None
    extra_A:	300 × 6
      obsm:	'spatial'
      layers:	None, 'weighted'
      obsp:	'spatial_connectivities'
    extra_B:	300 × 6
      obsm:	'spatial'
      layers:	None, 'weighted'
      obsp:	'spatial_connectivities'
    extra_C:	300 × 6
      obsm:	'spatial'
      layers:	None, 'weighted'
      obsp:	'spatial_connectivities'
views: dict_keys(['intra', 'extra_A', 'extra_B', 'extra_C'])
intra features: ['recepE', 'recepF', 'recepG']
extra_B features: ['ligA', 'ligB', 'ligC', 'ligD', 'ligX', 'ligY']
extra_C features: ['ligA', 'ligB', 'ligC', 'ligD', 'ligX', 'ligY']
receiver groups: ['A', 'B', 'C']


,target,intra_group,intra_R2,multi_R2,gain_R2,intra,extra_B,extra_C,receiver_celltype
0,recepE,A,0.738150,0.971999,0.233849,0.088693,0.634861,0.276446,A
1,recepF,A,0.715522,0.935123,0.219600,0.288307,0.711693,0.000000,A
2,recepG,A,0.097875,0.856170,0.758295,0.234957,0.000000,0.765043,A


,target,predictor,intra_group,view,importances,receiver_celltype,sender_celltype
26,recepE,ligA,A,extra_B,40.295260,A,B
27,recepE,ligX,A,extra_B,-4.199503,A,B
30,recepF,ligB,A,extra_B,41.292515,A,B
42,recepF,ligB,A,extra_C,3.988477,A,C
43,recepF,ligD,A,extra_C,5.687730,A,C
44,recepF,ligY,A,extra_C,-2.901657,A,C
47,recepG,ligC,A,extra_C,24.743411,A,C


All reported interactions are resource-supported.


## Constructing and fitting the object

The constructor creates an `intra` view with the receptor features and one
sender-specific extra view for each cell type. In this example, the boolean
column `is_A` selects A cells as the receiver group during fitting.

During `__call__`, `exclude_autocrine=True` excludes the extra view matching
the current receiver cell type. For this simulation, A does not express the
sender ligands, but `extra_A` is still excluded explicitly. For each
receptor target, only ligands paired with that receptor in `resource_sc` are
used before fitting. Consequently, unsupported ligand-receptor pairs cannot
influence the learned model.

`target_view` keeps track of which views are active for each target because
different targets can have different valid sender views. The results contain
one row per receptor target for receiver A, and the interaction table also
annotates the sender cell type. The final assertion checks that every
reported extra-view interaction is supported by the resource.
